In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1) SHAP용 입력 데이터 준비
# ------------------------------------------------------------
# 목적:
# - 이미 학습된 최종 모델(predictor)에 대해
#   글로벌 설명(global explanation)과
#   로컬 설명(local explanation)을 생성하기 위한 입력 데이터를 준비한다.
#
# 사용 데이터:
# - train_data: 모델 학습 구간 데이터
# - test_data: 최종 평가 구간 데이터
# - dashboard_df: test_data 기반으로 생성된 대시보드용 결과 테이블
#
# 기본 원칙:
# - SHAP 계산은 비용이 크기 때문에 전체 데이터를 모두 쓰지 않고
#   일부 샘플(background / global sample / high-risk sample)을 사용한다.
# ============================================================

# SHAP background dataset용 학습 데이터
# - final_features만 사용한다.
# - SHAP masker / explainer가 "기준 분포"로 참조할 데이터다.
X_train_bg = train_data[final_features].copy()

# 테스트 구간 전체 데이터 중
# - user_id는 나중에 어떤 사용자 설명인지 식별하기 위해 유지
# - final_features는 실제 SHAP 계산 입력
X_test_full = test_data[["user_id"] + final_features].copy()

# background sample 생성
# - 학습 데이터 전체를 쓰지 않고 최대 100건만 샘플링
# - random_state=42로 고정하여 재현 가능성 확보
# - 데이터가 100건보다 적으면 전체 사용
background = X_train_bg.sample(n=min(100, len(X_train_bg)), random_state=42)

# 글로벌 SHAP 계산용 샘플 생성
# - test 데이터 전체를 다 돌리지 않고 최대 200건만 샘플링
# - global explanation은 전체 경향 확인 목적이므로 대표 샘플 일부만 사용
# - 최종적으로 final_features 컬럼만 남김
X_global = X_test_full.sample(n=min(200, len(X_test_full)), random_state=42)[final_features].copy()

# High Risk 사용자 중 상위 5명 추출
# - dashboard_df에는 이미 churn_probability_pct와 Risk_Band가 붙어 있음
# - 그 중 Risk_Band == "High"인 사용자만 대상
# - churn_probability_pct 내림차순으로 정렬 후 상위 5명 선택
# - 이후 로컬 SHAP 설명 대상 사용자 집합으로 사용
high_users = (
    dashboard_df.loc[dashboard_df["Risk_Band"] == "High",
                     ["user_id", "Risk_Band", "churn_probability_pct"]]
    .sort_values("churn_probability_pct", ascending=False)
    .head(5)
    .copy()
)

# high_users와 test feature 데이터를 user_id 기준으로 결합
# - 어떤 high-risk 사용자의 어떤 feature 값이었는지 복원
# - 로컬 SHAP 결과를 사용자 메타정보와 함께 해석하기 위해 필요
X_high = high_users.merge(X_test_full, on="user_id", how="left")

# 로컬 SHAP 계산에는 모델 입력 feature만 사용
X_high_features = X_high[final_features].copy()

# ============================================================
# 2) SHAP용 예측 함수 정의
# ------------------------------------------------------------
# shap.Explainer는 내부적으로 "입력 X를 받아 예측값을 반환하는 함수"를 필요로 한다.
# 여기서는 양성 클래스(= churn, target=1)의 예측 확률을 반환하는 함수를 정의한다.
#
# 처리 방식:
# - 입력이 numpy array이면 DataFrame으로 복원
# - final_features 컬럼 순서 보장
# - predictor.predict_proba(...).iloc[:, 1] 로 양성 클래스 확률 반환
# ============================================================
def predict_positive_proba(X):
    # SHAP 내부 계산 과정에서는 입력이 numpy.ndarray로 들어올 수 있음
    # 이 경우 feature 이름이 사라지므로 DataFrame으로 다시 변환
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=final_features)

    # 최종 feature 순서를 보장한 뒤
    # 양성 클래스(1 = churn) 확률만 numpy 배열 형태로 반환
    return predictor.predict_proba(X[final_features]).iloc[:, 1].to_numpy()

# ============================================================
# 3) Explainer 생성
# ------------------------------------------------------------
# masker:
# - SHAP이 feature masking / perturbation 시 참조하는 background 분포
# - Independent masker를 사용하여 각 feature를 독립적으로 처리
#
# explainer:
# - permutation 알고리즘 기반 설명기
# - 모델이 트리 전용 SHAP을 직접 지원하는 방식이 아니라
#   "예측 함수" 기반으로 일반화된 설명기를 사용하는 구조
# ============================================================

# background 데이터를 기반으로 independent masker 생성
masker = shap.maskers.Independent(background)

# 예측 함수 + masker를 사용하여 SHAP explainer 생성
# algorithm="permutation"은 모델 예측값 변화량을 기반으로 SHAP 값을 추정하는 방식
explainer = shap.Explainer(
    predict_positive_proba,
    masker=masker,
    algorithm="permutation"
)

# SHAP 계산 시 사용할 평가 횟수 설정
# - feature 수가 적을 때 너무 작아지지 않도록 최소 80 보장
# - 보통 permutation explainer는 feature 수에 비례한 eval 수가 필요
max_evals = max(2 * len(final_features) + 1, 80)

# ============================================================
# 4) 글로벌 SHAP 계산
# ------------------------------------------------------------
# 목적:
# - 전체적으로 어떤 feature가 churn probability 예측에
#   많이 기여하는지 파악
#
# 산출물:
# - global_importance.csv : 평균 절대 SHAP값 기준 변수 중요도 테이블
# - beeswarm plot png     : 분포/방향성을 함께 보는 시각화
# ============================================================

# 글로벌 샘플(X_global)에 대한 SHAP 계산
# - 각 행/각 변수별 SHAP value 생성
shap_global = explainer(X_global, max_evals=max_evals)

# 평균 절대 SHAP값 기준 글로벌 중요도 테이블 생성
# - feature: 변수명
# - mean_abs_shap: 절대 SHAP값 평균
#   => 전체적으로 영향력이 큰 변수를 위로 정렬
global_importance = pd.DataFrame({
    "feature": final_features,
    "mean_abs_shap": np.abs(shap_global.values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

# 글로벌 중요도 테이블 저장
# - 후속 보고서/README/대시보드 참고용 산출물
global_importance.to_csv(SHAP_GLOBAL_CSV, index=False)

# Beeswarm plot 저장
# - 각 feature의 SHAP 분포와 영향 방향을 함께 볼 수 있는 대표 시각화
# - max_display=len(final_features)로 전체 변수 모두 표시
# - show=False로 화면 출력 대신 파일 저장용으로 사용
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_global, max_display=len(final_features), show=False)
plt.tight_layout()
plt.savefig(SHAP_BEESWARM_PNG, dpi=200, bbox_inches="tight")
plt.close()

# ============================================================
# 5) 로컬 SHAP 계산 (High Risk 상위 유저)
# ------------------------------------------------------------
# 목적:
# - High Risk 사용자 중에서도 churn probability가 높은 상위 사용자에 대해
#   "왜 이 사용자가 높은 위험도로 분류되었는가"를 개별 수준에서 설명
#
# 방식:
# - X_high_features에 대해 SHAP 계산
# - 각 사용자별 SHAP value 중 양(+)의 기여를 한 변수만 추출
# - 그중 상위 3개를 주요 이탈 사유(top reasons)로 저장
# ============================================================

# High Risk 상위 사용자들에 대한 로컬 SHAP 계산
shap_high = explainer(X_high_features, max_evals=max_evals)

# 내부 변수명을 대시보드/보고서용 한글 라벨로 변환하기 위한 매핑
# - local reason table 생성 시 함께 사용
feature_label_map = {
    "had_core_watch_history_rebuilt": "핵심 콘텐츠 시청 이력",
    "watch_hours": "전체 시청량",
    "content_diversity_score": "콘텐츠 소비 다양성",
    "price_score": "가격 민감도",
    "had_watch_delta_rebuilt": "최근 시청 변화",
    "days_since_last_watch": "휴면 신호",
    "freq_smartphone": "스마트폰 이용 패턴",
    "freq_tv_set": "TV 이용 패턴",
    "completion_rate": "완주율",
    "search_engagement": "검색 참여도"
}

# 사용자별 로컬 설명 결과를 담을 리스트
reason_rows = []

# High Risk 상위 사용자 각각에 대해 설명 테이블 생성
for i in range(len(X_high)):
    # 현재 사용자 식별 정보 추출
    user_id = X_high.iloc[i]["user_id"]
    risk_band = X_high.iloc[i]["Risk_Band"]
    churn_prob = X_high.iloc[i]["churn_probability_pct"]

    # 현재 사용자에 대한 feature별 SHAP 값
    shap_vals = shap_high.values[i]

    # 현재 사용자의 실제 feature 값
    feat_vals = X_high_features.iloc[i]

    # SHAP 값이 큰 순서대로 정렬
    # 값이 클수록 churn 확률을 높이는 방향 기여가 큼
    order = np.argsort(shap_vals)[::-1]

    # 양(+)의 SHAP 값만 선택
    # 즉, churn risk를 올리는 방향으로 기여한 feature만 추림
    # 그중 상위 3개까지만 사용
    positive_idx = [j for j in order if shap_vals[j] > 0][:3]

    # 사용자 단위 결과 행 초기화
    row = {
        "user_id": user_id,
        "Risk_Band": risk_band,
        "churn_probability_pct": churn_prob
    }

    # 상위 양의 기여 변수들을 rank별로 저장
    # 저장 항목:
    # - 원래 feature명
    # - 한글 라벨
    # - 해당 사용자의 실제 변수값
    # - 해당 변수의 SHAP 값
    for rank, j in enumerate(positive_idx, start=1):
        row[f"reason_{rank}_feature"] = final_features[j]
        row[f"reason_{rank}_feature_label"] = feature_label_map.get(final_features[j], final_features[j])
        row[f"reason_{rank}_feature_value"] = feat_vals.iloc[j]
        row[f"reason_{rank}_shap"] = shap_vals[j]

    # 사용자별 결과를 리스트에 누적
    reason_rows.append(row)

# 누적된 사용자별 로컬 설명 결과를 데이터프레임으로 변환
local_reason_df = pd.DataFrame(reason_rows)

# 상위 3개 이유의 한글 라벨을 합쳐서 요약 텍스트 생성
# 예:
# "휴면 신호, 전체 시청량, 검색 참여도"
# 대시보드 표/카드형 UI에서 바로 보여주기 쉽게 만든 요약 컬럼
local_reason_df["top_reason_text"] = local_reason_df.apply(
    lambda row: ", ".join(
        [row[f"reason_{k}_feature_label"] for k in [1, 2, 3]
         if pd.notna(row.get(f"reason_{k}_feature_label"))]
    ),
    axis=1
)

# High Risk 상위 사용자 로컬 설명 결과 저장
# - 사용자별 주요 이탈 사유와 SHAP 기여값을 담은 테이블
local_reason_df.to_csv(SHAP_HIGH_RISK_CSV, index=False)